# 04 - TRL DPOTrainer + LoRA persona tuning

Accelerator: GPU T4 x2, Internet on. Trains a LoRA adapter on HuggingFaceTB/SmolLM2-360M-Instruct with
`data/persona_lora/ramsay_style_preferences.json`, then generates on the fixed persona evaluation set.

In [ ]:
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO = "small-model-alignment"
GITHUB_USER = "ahmedamirkol01-art"
os.chdir("/kaggle/working")
if not os.path.exists(REPO):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    result = subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git"], capture_output=True, text=True)
    print("cloned" if result.returncode == 0 else result.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", f"https://github.com/{GITHUB_USER}/{REPO}.git"])
os.chdir(f"/kaggle/working/{REPO}")
os.environ["PYTHONPATH"] = "."
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!git log --oneline -3

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl==0.29.1"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("visible GPUs:", torch.cuda.device_count())

In [ ]:
!python -m pytest -q tests/

Pilot run.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --output_dir outputs/lora_pilot --max_steps 3
!rm -rf outputs/lora_pilot

Full run.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/train_lora_dpo.py --output_dir outputs/lora_persona

Sanity check on 5 prompts, then the full persona evaluation set for the base instruct model and the LoRA-DPO model.

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_persona/adapter --use_chat_template --eval_set eval_sets/persona_style_eval.json --generator smollm2-360m-lora-dpo --output_path outputs/sanity/persona_lora.json --limit 5

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --use_chat_template --eval_set eval_sets/persona_style_eval.json --generator smollm2-360m-instruct --output_path outputs/generations/persona_smollm2_instruct.json
!CUDA_VISIBLE_DEVICES=0 python scripts/generate.py --model_path HuggingFaceTB/SmolLM2-360M-Instruct --adapter_path outputs/lora_persona/adapter --use_chat_template --eval_set eval_sets/persona_style_eval.json --generator smollm2-360m-lora-dpo --output_path outputs/generations/persona_smollm2_lora_dpo.json
!python scripts/build_pairs.py --candidate_path outputs/generations/persona_smollm2_lora_dpo.json --baseline_path outputs/generations/persona_smollm2_instruct.json --output_path outputs/eval/persona_lora_vs_instruct_pairs.jsonl

In [ ]:
!python scripts/plot_metrics.py --out_dir outputs/plots
!cd /kaggle/working && zip -r -q lora_run_outputs.zip small-model-alignment/outputs
!ls -lh outputs/lora_persona outputs/generations outputs/eval